In [0]:
import pandas as pd
import json
import time
import os
from datetime import datetime

# ---------------------------------------------------------
# 1. Read CSV
# ---------------------------------------------------------
# csv_path = "/Workspace/Users/databelli14@gmail.com/÷

csv_path = '/Workspace/Users/kishorecoder685@gmail.com/fraud_watchlist_file_generator/fraud_watchlist.csv'

df = pd.read_csv(csv_path, dtype=str)

# Make sure entity_id is always a string
df["entity_id"] = df["entity_id"].astype(str)

print(f"Total rows in CSV: {len(df)}")


# ---------------------------------------------------------
# 2. Target output path
# ---------------------------------------------------------
output_path = "/Volumes/finguard/source/fraud_watchlist/source_data/"

dbutils.fs.mkdirs(output_path)


# ---------------------------------------------------------
# 3. Process every row with a 5-second interval
# ---------------------------------------------------------
for index, row in df.iterrows():

    print("=" * 80)
    print(f"Reading CSV row {index + 1} of {len(df)}")
    print(f"watchlist_id: {row['watchlist_id']}")
    print(f"entity_id:    {row['entity_id']}")
    print(f"risk_level:   {row['risk_level']}")

    # Convert the row to a dictionary
    watchlist_record = row.to_dict()

    # -----------------------------------------------------
    # 4. Create JSON from the current CSV row
    # -----------------------------------------------------
    json_data = json.dumps(
        watchlist_record,
        indent=2,
        ensure_ascii=False
    )

    # -----------------------------------------------------
    # 5. Generate JSON filename
    # -----------------------------------------------------
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

    watchlist_id = row["watchlist_id"]

    file_name = f"fraud_watchlist_{watchlist_id}_{timestamp}.json"

    full_path = output_path + file_name

    # -----------------------------------------------------
    # 6. Write JSON to Volume
    # -----------------------------------------------------
    dbutils.fs.put(
        full_path,
        json_data,
        overwrite=False
    )

    print(f"Created: {full_path}")

    # -----------------------------------------------------
    # 7. Wait 5 seconds before reading next row
    # -----------------------------------------------------
    if index < len(df) - 1:
        print("Waiting 5 seconds before reading next row...")
        time.sleep(5)


print("=" * 80)
print("Finished processing all CSV rows.")